# 06: Institutional Model Explainability, SHAP Attribution & Feature Selection

This interactive research notebook demonstrates the **MDK Trading Oracle Explainability & Feature Selection Engine** across **Models 1, 2, and 3**.

---

### Core Objectives
1. **Pillar 1: Local Signal Explainability (Live $T+1$)**:
   - Decomposes the upcoming session's forecast into an exact **Additive Waterfall**:
     $$\hat{y} = \mathbb{E}[y] + \sum_{i=1}^M \phi_i$$
     where $\mathbb{E}[y]$ is the baseline expected market flow, and $\phi_i$ is the exact TL (or return %) contribution of feature $i$.
2. **Pillar 2: Semantic Microstructure Cluster Rollup**:
   - Instead of 45 raw features causing cognitive overload, features are automatically rolled up into **institutional clusters** (*Closing Momentum*, *Competitor Deltas*, *Macro Rates*, *Tertip Inventory*, etc.) to reveal where true alpha originates.
3. **Pillar 3: Data-Driven Feature Pruning & Collinearity Screening**:
   - Uses **Out-of-Sample Permutation Importance** and **Pairwise Correlation Screening** ($|r| \ge 0.85$) to flag redundant features and recommend updates for `config/features.yaml`.


## 1. Setup & Lakehouse Connection

We connect to the local DuckDB lakehouse strictly in **read-only mode** (`read_only=True`) to ensure zero lock contention with active pipeline runs.


In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML

from mdk_trading_oracle.core.db import DuckDBManager
from mdk_trading_oracle.models.features_config import FeatureSelector
from mdk_trading_oracle.models.day_start.forecaster import DayStartForecaster
from mdk_trading_oracle.models.sector_day_start.forecaster import SectorDayStartForecaster
from mdk_trading_oracle.models.stock_reaction.forecaster import StockReactionForecaster
from mdk_trading_oracle.explainability import (
    ModelExplainer,
    FeatureAuditor,
    plot_waterfall,
    plot_cluster_donut,
    format_markdown_card,
)

# Connect in read-only mode to prevent file lock conflicts
db = DuckDBManager(read_only=True)
print("Lakehouse connection established (read-only mode).")


Lakehouse connection established (read-only mode).


## 2. Model 1 (Macro Day-Start): Live $T+1$ Signal Decomposition

Here we generate the live upcoming forecast for **Bank of America (MLB)** and decompose the predicted net flow into its exact additive catalysts and headwinds.


In [ ]:
# Instantiate forecaster with read-only database connection
forecaster_m1 = DayStartForecaster(db=db)

# Compute live forecast for upcoming session T+1
res_m1 = forecaster_m1.forecast_next_day()

# Extract local explanation
exp_m1 = forecaster_m1.explain_forecast()

# Display formatted Markdown executive card
display(Markdown(format_markdown_card(exp_m1)))


### Interactive Prediction Waterfall Chart

The waterfall shows the exact step-by-step path:
- **Baseline Expected Market Value $\mathbb{E}[y]$**: What BofA typically does on an average morning.
- **Green Bars (Catalysts)**: Market signals that pushed BofA's predicted flow higher.
- **Red Bars (Headwinds)**: Signals that exerted downward drag on BofA's predicted flow.
- **Blue Bar (Final Forecast $\hat{y}$)**: The resulting predicted opening net flow.


In [ ]:
fig_waterfall_m1 = plot_waterfall(
    exp_m1,
    max_display=8,
    title=f"Model 1: BofA Day-Start Flow Decomposition ({res_m1.forecast_date})",
)
fig_waterfall_m1.show()


## 3. Global Microstructure Cluster Alpha Share

Across all historical sessions, which institutional feature clusters drive the majority of the model's predictive power?


In [ ]:
# Compute cross-session global feature importance
global_exp_m1 = forecaster_m1.explain_global()

# Display Donut Chart showing cluster share %
fig_donut_m1 = plot_cluster_donut(
    global_exp_m1,
    title="Macro Day-Start: Microstructure Cluster Alpha Share (%)",
)
fig_donut_m1.show()

# Display tabular ranking
display(Markdown("### Semantic Microstructure Cluster Scoreboard"))
display(global_exp_m1.cluster_importance_df)


## 4. Model 2 (Sector Day-Start): Sector-Level Allocation Decomposition

Select any tracked BIST sector to inspect what microstructure features drove BofA's predicted capital allocation for that specific sector.


In [ ]:
forecaster_m2 = SectorDayStartForecaster(db=db)

# Tracked liquid sectors
available_sectors = ["Banking", "Holding", "Transportation", "Energy & Refining", "Defense & Tech", "Retail Trade"]

def inspect_sector_explanation(sector_name):
    print(f"Generating live forecast & attribution for sector: {sector_name}...")
    exp_sec = forecaster_m2.explain_sector_forecast(sector=sector_name)
    if exp_sec:
        display(Markdown(format_markdown_card(exp_sec)))
        fig = plot_waterfall(exp_sec, max_display=6, title=f"BofA Day-Start Allocation: {sector_name}")
        fig.show()
    else:
        print(f"No explanation available for {sector_name}.")

# Interactive dropdown
dropdown = widgets.Dropdown(
    options=available_sectors,
    value="Banking",
    description="Sector:",
    disabled=False,
)
widgets.interact(inspect_sector_explanation, sector_name=dropdown);


## 5. Model 3 (Stock Intraday Reaction): Equity Return % Decomposition

Select an individual BIST 30 equity and reaction window ($W_2, W_3, W_5$) to inspect the microstructure drivers behind the forecasted return percentage.


In [ ]:
tracked_symbols = ["THYAO", "AKBNK", "GARAN", "EREGL", "TUPRS", "BIMAS", "ASELS", "KCHOL", "ISCTR", "YKBNK"]
window_options = [("Window 2: First Reaction (10:30-11:30)", "w2"),
                  ("Window 3: Midday Followup (11:30-14:30)", "w3"),
                  ("Window 5: Closing Session (16:00-18:15)", "w5")]

def inspect_stock_reaction(symbol, window_tuple):
    window_key = window_tuple
    print(f"Analyzing {symbol} ({window_key})...")
    forecaster_sr = StockReactionForecaster(symbol=symbol, window=window_key, db=db)
    res_sr = forecaster_sr.forecast_next_window(replace_active=False)
    
    if res_sr and res_sr.explanation:
        from mdk_trading_oracle.explainability.types import LocalExplanation, FeatureAttribution, ClusterAttribution
        d = res_sr.explanation
        exp = LocalExplanation(
            model_name=d["model_name"],
            model_version=d["model_version"],
            target_broker_or_symbol=d["target_broker_or_symbol"],
            base_value=d["base_value"],
            predicted_value=d["predicted_value"],
            unit=d.get("unit", "%"),
            top_positive_drivers=[FeatureAttribution(**item) for item in d.get("top_positive_drivers", [])],
            top_negative_drivers=[FeatureAttribution(**item) for item in d.get("top_negative_drivers", [])],
            cluster_attributions=[
                ClusterAttribution(
                    cluster_name=k,
                    total_attribution=v["total_attribution"],
                    total_abs_attribution=abs(v["total_attribution"]),
                    percentage_share=v["percentage_share"],
                    feature_count=1,
                    top_feature=v.get("top_feature", "")
                )
                for k, v in d.get("cluster_breakdown", {}).items()
            ]
        )
        display(Markdown(format_markdown_card(exp)))
        fig = plot_waterfall(exp, max_display=7, title=f"Stock Reaction Return % Waterfall: {symbol} ({window_key})")
        fig.show()
    else:
        print(f"Could not compute forecast/explanation for {symbol} ({window_key}).")

symbol_dd = widgets.Dropdown(options=tracked_symbols, value="THYAO", description="Symbol:")
window_dd = widgets.Dropdown(options=window_options, value="w2", description="Window:")
widgets.interact(inspect_stock_reaction, symbol=symbol_dd, window_tuple=window_dd);


## 6. Systematic Feature Selection & Pruning Audit

Run the **FeatureAuditor** to evaluate out-of-sample alpha contributions, detect collinear feature pairs ($|r| \ge 0.85$), and identify prune candidates.


In [ ]:
# Run Feature Selection Audit on Model 1
report = forecaster_m1.audit_features(collinearity_threshold=0.85)

display(Markdown(f"""
### Feature Audit Summary: `{report.model_name}`
- **Evaluated Historical Sessions**: {report.evaluated_sessions}
- **Active Features Evaluated**: {report.total_features}
- **Collinear Pairs Detected**: {len(report.collinear_pairs)}
- **Prune Candidates Recommended**: {len(report.prune_candidates)}
"""))

# Top 10 Alpha Drivers
if report.top_drivers:
    display(Markdown("#### Top 10 Out-of-Sample Alpha Drivers"))
    display(pd.DataFrame(report.top_drivers))

# Collinear Pairs
if report.collinear_pairs:
    display(Markdown("#### Collinear Feature Redundancies (|r| >= 0.85)"))
    display(pd.DataFrame(report.collinear_pairs))

# Prune Candidates Table
if report.prune_candidates:
    display(Markdown("#### Recommended Features to Exclude"))
    display(pd.DataFrame(report.prune_candidates))

# Generated YAML Snippet
if report.recommended_features_yaml:
    display(Markdown("#### Recommended `config/features.yaml` Snippet"))
    print(report.recommended_features_yaml)
